<a href="https://colab.research.google.com/github/salinela/carbon-portfolio-project-v2/blob/main/notebooks/08_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# set up: desktop
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import sqlite3
import time
import seaborn as sns

# path set up:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

from src.config import DATA_RAW, DATA_PROCESSED
from src import eda

# database connection set up:
DB = ROOT/'data/carbon.db'
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")

In [1]:
# set up: Google Colab
import sys
import sqlite3
import time
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:

# mount drive (data artifacts live here — never in git)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# clone fresh, or pull if it already exists (so re-running the cell doesn't error)
import os
REPO = "/content/repo"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/salinela/carbon-portfolio-project-v2.git {REPO}
%cd {REPO}

Cloning into '/content/repo'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 207 (delta 124), reused 112 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 337.22 KiB | 6.13 MiB/s, done.
Resolving deltas: 100% (124/124), done.
/content/repo


In [4]:
# root on path — mirrors desktop's package-style imports
sys.path.insert(0, REPO)
print("root on path:", REPO)

root on path: /content/repo


In [5]:
# live-reload src edits after a git pull without restarting the runtime
from src import eda
from src import feature_engineering as fe
from src.config import DATA_RAW, DATA_PROCESSED

In [6]:
# DB copied to LOCAL disk (not the Drive FUSE mount) to avoid SQLite locking.
# Needed to read/query it, not just to rebuild — copy once per session.
DRIVE = "/content/drive/MyDrive/carbon_project_v2"
if not os.path.exists("/content/carbon.db"):
    !cp "{DRIVE}/carbon.db" /content/carbon.db

In [7]:
DB = "/content/carbon.db"
con = sqlite3.connect(DB, timeout=30)
con.execute("PRAGMA foreign_keys = ON;")

one-time usage

In [ ]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# one-time cleanup of the stale broken view
con.execute("DROP VIEW IF EXISTS v_company_emissions;")
con.commit()

# Phase 0: Data Assembly

In [8]:
# Build meta (summary dataframe/profile snapshot of all companies in the database) and meta_diag (summary dictionary of meta) for EDA:
meta, meta_diag = eda.build_meta(con)

# Build fy (yearly data: emissions and fundamentals) per company year:
fy,   fy_diag   = eda.build_firm_year(con)

meta_diag, fy_diag

({'n_companies': 8288,
  'by_universe': {'EU': 7988, 'ETS': 300},
  'eligible_n': 5883,
  'sector_nulls_master': 58,
  'sector_nulls_after_backfill': 58,
  'country_nulls': 0,
  'bvd_nulls': 0,
  'coverage_status_counts': {'mapped_loaded': 5883,
   'unmapped_exchange': 1773,
   'mapped_no_data': 523,
   'no_ticker': 109}},
 {'n_rows': 12487,
  'n_companies': 1306,
  'year_range': (2012, 2025),
  'by_source': {'trucost': 9214, 'ets_registry': 3273},
  'revenue_nulls': 573,
  'intensity_nulls': 573})

In [9]:
# summarising intial eligible companies:
fy_ids = set(fy["company_id"])
elig   = meta[meta["eligible"] == 1] # referring to no stock price series data available

mask   = elig.index.isin(fy_ids)
print("eligible:", len(elig))
print("eligible w/ emissions:", int(mask.sum()))
print(elig[mask]["universe"].value_counts().to_dict())

eligible: 5883
eligible w/ emissions: 1259
{'EU': 978, 'ETS': 281}


# Phase 1: Universe Characterization

### Section A: Compute carbon itensity tiers (source registry x 1-digit NACE x year)

In [10]:
# --- cohort flags on meta (enables optional carbon-blind comparison) ---
fy_ids = set(fy["company_id"])
meta["has_emissions_data"] = meta.index.isin(fy_ids).astype(int)
meta["carbon_sample"] = ((meta["eligible"] == 1) &
                         (meta["has_emissions_data"] == 1)).astype(int)

# cross-check against master's has_emissions flag:
print("has_emissions agree:",
      (meta["has_emissions"] == meta["has_emissions_data"]).mean())
print("carbon_sample n:", int(meta["carbon_sample"].sum()))

has_emissions agree: 0.9639237451737451
carbon_sample n: 1259


In [11]:
# --- nace1 for tiering: master, backfilled from orbis, first digit ---
nace_full = meta["nace_code"].fillna(meta["orbis_nace_code"])
nace1 = nace_full.astype("string").str.extract(r"(\d)")[0]   # extract nace1 code (first nace digit onky), index = company_id

# --- compute tiers using eda.compute_tiers on read; create carbon_tier at fy
fy, tier_diag = eda.compute_tiers(fy, nace1) # carbon_tier stored in fy
tier_diag

{'tier_counts': {'non_ets_high': 2947,
  'non_ets_low': 2911,
  'non_ets_medium': 2872,
  'ets_high': 1074,
  'ets_low': 1044,
  'ets_medium': 1015,
  <NA>: 573,
  'ets_untiered': 41,
  'non_ets_untiered': 10},
 'nace1_nulls': 0,
 'n_untiered': 51,
 'n_tiered': 11863}

In [12]:
# --- diagnose the difference between has_emission (master table in carbon.db) and has_emission_data (rejoined with emissions table):
d = meta[meta["has_emissions"] != meta["has_emissions_data"]]
print(len(d))
print(d.groupby(["has_emissions", "has_emissions_data"]).size())
print(d["universe"].value_counts().to_dict())

299
has_emissions  has_emissions_data
0              1                     299
dtype: int64
{'ETS': 299}


### Section B: Monthly Tier-based Portfolio Returns Helper

Main objective: for each month, take every firm currently sitting per tier and average their forward returns (equal-weighted basket)

tier_portfolio_returns produces one such series per tie; "do high-carbon baskets earn different returns than low-carbon ones"

The attach_tier_asof step is what tells each company-month which basket it was in at that date, using the 1-July lag so you're never using an emissions figure before it was public.

**Profiling Missingness in Monthly Returns Data**

We have two monthly price series dataframes a) **features** consisting of a suite of features derived from OHLVC data and c) **label** consisting of monthly forward returns; each monthly data comes with a carbon tier if emissions data are available

We first profile if there is any systematic NAs for specific month-end dates and provide a corresponding fix. Source of missing data:
- emission data --> untiered carbon status (likely for edge effects from too early/later dates but unlikely for in-between months)
- too little company in each bucket (low, medium, high)
- no monthly returns data
- no feature data

How do spot if there is missing data?
- see how many companies were in each bucket per year/month (if there is any thin cross-section)
- flag months with any missing data
- flag if the problem arise from no carbon data or returns data

Fix:
- understand the sources of missingness (specific dates: country-specific holidays --> re-deriving data for these dates)
paired dates (previous missing month data carries over to the next month)
- merge with existing data to form new and more complete table


**Desired outcome: No missingness per month (unless edge out-of-range months)**

In [13]:
# panel (features and label):
features = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/features_month_end.parquet")
label    = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/label_fwd_return.parquet")

In [14]:
# pivot to wide:
panel = features.pivot_table(index=["company_id", "date"],
                             columns="signal_name", values="value")

# join features with label:
panel = panel.join(label.set_index(["company_id", "date"])["fwd_ret"])
print("panel:", panel.shape)

panel: (730161, 42)


In [15]:
# ---- step 2: Attach tiers as-of ----

# Attach carbon tiers to each monthly features panel:
panel_t = eda.attach_tier_asof(panel, fy)

# Attached tier NA rates across the whole data:
print("Tier coverage (monthly data with carbon tiers):", round(panel_t["carbon_tier"].notna().mean(), 3))

Tier coverage (monthly data with carbon tiers): 0.184


In [16]:
# Crude diagnostic reveal that low coverage arise from very little companies having carbon data:
p = panel.reset_index()

# Number of initial companies in the panel (monthly features/label data)
n_panel = p['company_id'].nunique()
n_fy = fy[fy['carbon_tier'].notna()]['company_id'].nunique()

# Companies with tier data compared to companies with features/label data:
print("Proportion of companies with carbon tier data compared to total companies with panel data: ",round(n_fy/n_panel,3))

Proportion of companies with carbon tier data compared to total companies with panel data:  0.215


In [18]:
# Calculate averaged monthly returns per tier-month (equal-weighted, arithmetic); tier-dates with < min firms are NaN
tret = eda.tier_portfolio_returns(panel_t)


# Initial diagnostics (NA rates for monthly returns across the months per tier):
print(tret.isna().mean())

carbon_tier
ets_high            0.083871
ets_low             0.096774
ets_medium          0.096774
ets_untiered        0.935484
non_ets_high        0.077419
non_ets_low         0.090323
non_ets_medium      0.090323
non_ets_untiered    1.000000
dtype: float64


From the initial missing rates:
- untiered buckets have 100% missing rates for averaged monthly returns (too thin cross section slice, pertaining to edge industries with insufficient peers to do within-industry carbon tiering (fall back to NAs)


_untiered vs NA for carbon_tier are conceptually distinct:
- NA = firm-year had no intensity (missing revenue → couldn't even attempt a tercile). 573 of these.
- _untiered = firm-year had intensity but sat in a too-thin cohort, about 51 of these

In [26]:
# ---- step 3: FLAG — month-ends with abnormally low return coverage ----
cov = eda.label_coverage_scan(features, label)          # long v1 frames
flagged = cov[cov["flag"] == True]
flagged

,n_features,n_label,ratio,flag
date,,,,
2026-06-29,5820,0.0,0.000000,True
2024-02-29,5563,247.0,0.044401,True
2024-03-29,1929,248.0,0.128564,True
2018-02-28,4176,727.0,0.174090,True
2013-02-28,3287,576.0,0.175236,True
2018-03-30,2444,728.0,0.297872,True
2013-03-29,1847,578.0,0.312940,True
2021-11-30,5092,2160.0,0.424195,True
2024-11-29,5655,2431.0,0.429885,True


**Diagnostics**
- ratio refers to ratio of n_label over n_features (we base on whether n_label is abnormally low compared to other data available that day)
- flagged dates are consecutive (how returns are derived_
- certain dates are repeated (e.g., 28/2 and 29/3) --> 2024-03-29 is good friday (exchange closed and no data available for certain exchanges/tickers; This affects the calculations for 2024-02-29)


A 1-month-forward return at end-Feb needs a price at end-Mar; end-March 2018 and end-March 2024 are Good Friday (2018-03-30, 2024-03-29 — markets shut). So:

At end-Feb, fwd_ret looks one month ahead to a closed Good Friday → NaN.
At end-Mar (the Good Friday itself), there's no price today → NaN.

That's why it's always a Feb/Mar pair, and only in years where Good Friday lands on the month-end. It's a forward-looking gap in the target, not a hole in the features. Which is why widening the window never helped — the NaN is baked into how the label was constructed.

In [28]:
# ---- step 4: PROBE — tier-side or return-side gap? ----
# drop the final month: it's the true no-future edge, not a holiday
flag_dates = [d for d in flagged.index if d != flagged.index.max()]
probe = eda.probe_missing_month(panel_t, flag_dates)
probe

,n_firms,n_tier,n_ret,n_tier_and_ret,tier_rate,ret_given_tier,diagnosis
date,,,,,,,
2013-02-28,3287,0,572,0,0.000,NaN,tier/coverage gap
2013-03-29,1847,0,574,0,0.000,NaN,tier/coverage gap
2018-02-28,4176,989,723,55,0.237,0.056,tier/coverage gap
2018-03-30,2444,564,724,55,0.231,0.098,tier/coverage gap
2020-11-30,4680,1135,2058,538,0.243,0.474,tier/coverage gap
2020-12-31,4248,1068,2070,538,0.251,0.504,tier/coverage gap
2021-11-30,5092,1157,2120,496,0.227,0.429,tier/coverage gap
2021-12-31,4670,1091,2136,496,0.234,0.455,tier/coverage gap
2024-02-29,5563,1143,246,11,0.205,0.010,tier/coverage gap


In [ ]:
# ---- step 5: DIAGNOSE (read the probe) ----
# expect diagnosis == "label/return gap (holiday?)": tiers intact, returns collapsed.
# cross-check the dates land on Good Friday / year-end -> exchange-divergence confirmed.

In [29]:
# ---- step 6: FIX AT SOURCE + validate ----
label_v2 = fe.forward_return_label(con, horizon=1, kind="log")   # per-firm month_end_close

# merge old and new returns:
j = label.merge(label_v2, on=["company_id", "date"], suffixes=("_old", "_new"))

# see how many values changed?:
print("existing values changed:",
      int((~np.isclose(j["fwd_ret_old"], j["fwd_ret_new"], atol=1e-9)).sum()), "of", len(j))

# new rows added:
print("net new firm-month labels:", len(label_v2) - len(label))
label_v2.to_parquet(f"{DRIVE}/label_forward_return_v2.parquet")

existing values changed: 0 of 665683
net new firm-month labels: 73779


No new values should change, we are essentially adding data to old month ends previously missed due to bug in derivation pipeline

In [30]:
# ---- step 7: SNAP to month-end key (both sides) + assert invariant ----
features_me = eda.snap_to_month_end(features)
label_me    = eda.snap_to_month_end(label_v2)

eda.assert_month_end(features_me)
eda.assert_month_end(label_me)

In [32]:
# ---- step 8: rebuild + re-validate ----

# pivot panel to wide again and attach carbon tiers
panel = features_me.pivot_table(index=["company_id", "date"],
                                columns="signal_name", values="value")
panel = panel.join(label_me.set_index(["company_id", "date"])["fwd_ret"])
panel_t = eda.attach_tier_asof(panel, fy)

# re-average monthly returns by tiers:
tret = eda.tier_portfolio_returns(panel_t)

# drop edge months (tails with no data) and drop untiered companies:
win = tret.loc["2014-07-30":"2025-06-30"].drop(
    columns=["ets_untiered", "non_ets_untiered"], errors="ignore")


print(win.isna().mean())          # expect all 0.0

carbon_tier
ets_high          0.0
ets_low           0.0
ets_medium        0.0
non_ets_high      0.0
non_ets_low       0.0
non_ets_medium    0.0
dtype: float64


In [33]:
cov2 = eda.label_coverage_scan(
    features_me.assign(date=features_me["date"].dt.strftime("%Y-%m-%d")),
    label_me.assign(date=label_me["date"].dt.strftime("%Y-%m-%d")))
print(cov2[cov2["flag"]])         # expect only the final no-future month

            n_features  n_label  ratio  flag
date                                        
2026-06-30        5820      0.0    0.0  True


**Summary of derived data objects:**

**Persisted artifacts (Drive)**:

- **carbon.db** — SQLite source of truth: prices (daily OHLCV), fundamentals, company_emissions, ets_company_extras, master_company_list, orbis_core, installations, market_factors, ff_factors, symbol_coverage, etc.
- **features_month_end.parquet** — long (company_id, date, signal_name, value); the 41 price/technical signals on the month-end grid, ~2013–2026.
- **label_forward_return_v2.parquet** — long (company_id, date, fwd_ret); 1-month-forward log return, holiday-fixed. This is canonical; v1 is superseded.

**In-memory frames (rebuilt each session from the above)**:

- **meta** — one row per company (8,288); static attributes: universe, sector, country, listing_status, cohort flags (eligible, has_emissions_data, carbon_sample). Composition join key.
- **fy (firm_year)** — one row per company-year (~12,487); scope1_emissions, revenue, intensity, source, nace1, and the recomputed six-way carbon_tier. Feeds tiering + the grouped-mean grid.
- **panel** — company-month × 41 signals + fwd_ret, on the month-end key. The modeling/EDA matrix.
- **panel_t** — panel + as-of carbon_tier (1-July lag). Cross-sectional tier analysis.
- **tret** — monthly date × 6 tier columns; equal-weighted arithmetic tier-portfolio returns. Canonical window 2014-07-30 : 2025-06-30.

## Section C: Composition Counts

"know your sample" check

Describes who is actually in the panel and how that mix moves over time, so every later result (tier spreads, modeling, backtests) can be read against a known base and known thin spots.

Main questions:

- How many firms per year? Panel presence by year — is the universe growing, roughly balanced, or entry/exit-skewed?
- How does it split by universe/source? ETS vs non-ETS counts over time.
- How does it distribute by sector (NACE section) and country?
- How does it split across the six carbon tiers, per year — and are all six tiers thick enough (the ≥5-firm rule that already pins the usable 2014–2025 window)?
- Where are the thin cells? Which year × tier (and year × sector) combinations are too sparse to support cross-sectional claims downstream.
- Is there composition drift that could confound tier effects? e.g. the known Trucost large-cap / developed-market skew on the non-ETS side shifting across years.